In [10]:
%pip install tiktoken regex

Note: you may need to restart the kernel to use updated packages.


In [11]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [1]:
import tiktoken
import regex as re


In [2]:
with open("taylorswift.txt", "r", encoding="utf-8") as f:
    text = f.read()

In [3]:
tokens = text.encode("utf=8")
tokens = list(map(int, tokens))

In [4]:
# my solution
new_enc = {}
def BPEncode(tokens):
    d = {}
    for pair in zip(tokens, tokens[1:]):
        d[pair] = d.get(pair, 0) + 1

    if not d or max(d.values()) == 1:
        return tokens

    max_value = max(d.values())
    key = max(d, key=d.get) 

    max_token = max(tokens) + 1
    new_enc[max_token] = key

    new_tokens = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == key:
            new_tokens.append(max_token)
            i += 2
        else:
            new_tokens.append(tokens[i])
            i += 1

    return bytePairEncode(new_tokens)

In [5]:
class BasicTokenizer:
    
    def train(self, text, vocab_size, verbose=False):
        def get_stats(ids):
            counts = {}
            for pair in zip(ids, ids[1:]):
                counts[pair] = counts.get(pair, 0) + 1
            return counts
        
        def merge(ids, pair, idx):
            newids = []
            i = 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
                    newids.append(idx)
                    i += 2
                else:
                    newids.append(ids[i])
                    i += 1
            return newids


        tokens = text.encode("utf=8")
        tokens = list(map(int, tokens))
        
        max_num_in_vocab = 0
        for k in tokens:
            if k > max_num_in_vocab:
                max_num_in_vocab = k
        
        
        num_merges = vocab_size - max_num_in_vocab
        ids = list(tokens)
        
        self.merges = {}
        for i in range(num_merges):
            stats = get_stats(ids)
            pair = max(stats, key=stats.get)
            idx = max_num_in_vocab + 1 + i
            ids = merge(ids, pair, idx)
            self.merges[pair] = idx

    def encode(self, text):
        tokens = list(text.encode("utf=8"))
        
        for pair in self.merges:
            i = 0
            new_tokens = []
            while i < len(tokens):
                if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
                    new_tokens.append(self.merges.get(pair))
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1

            tokens = new_tokens
        return tokens
        
    def decode(self, tokens):
        for key, val in reversed(self.merges.items()):
            i = 0
            new_tokens = []
            while i < len(tokens):
                if tokens[i] == val:
                    new_tokens.append(key[0])
                    new_tokens.append(key[1])
                    i += 1
                else:
                    new_tokens.append(tokens[i])
                    i += 1

            tokens = new_tokens
            
        byte_data = bytes(tokens)
        return byte_data.decode('utf-8', errors='replace')

In [6]:
class RegexTokenizer:
    def __init__(self):
        GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
        self.pat = re.compile(GPT4_SPLIT_PATTERN)


    def train(self, text, vocab_size, verbose=False):
        
        def get_stats(text):
            counts = {}
            for word in text:
                for pair in zip(word, word[1:]):
                    counts[pair] = counts.get(pair, 0) + 1
            return counts
        
        def merge(word, pair, idx):
            new_word = []
            i = 0
            while i < len(word):
                if i < len(word) - 1 and word[i] == pair[0] and word[i+1] == pair[1]:
                    new_word.append(idx)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            return new_word

        text_chunks = re.findall(self.pat, text)
        ids = [list(chunk.encode("utf-8")) for chunk in text_chunks]
        
        num_merges = vocab_size - 256
        
        self.merges = {}
        for i in range(num_merges):
            stats = get_stats(ids)
            if not stats:
                print("stopped at: ", i, " merges")
                break
            pair = max(stats, key=stats.get)
            idx = 256 + i
            for i in range(len(ids)):
                ids[i] = merge(ids[i], pair, idx)
            self.merges[pair] = idx


    def encode(self, text):
        text_chunks = re.findall(self.pat, text)
        ids = [list(chunk.encode("utf-8")) for chunk in text_chunks]

        new_text = []
        for word in ids:
            for pair in self.merges:
                i = 0
                new_tokens = []
                while i < len(word):
                    if i < len(word) - 1 and (word[i], word[i+1]) == pair:
                        new_tokens.append(self.merges.get(pair))
                        i += 2
                    else:
                        new_tokens.append(word[i])
                        i += 1
    
                word = new_tokens
            new_text.append(word)
        flat_tokens = [tok for word in new_text for tok in word]
        return flat_tokens
        
    def decode(self, tokens):
        for key, val in reversed(self.merges.items()):
            i = 0
            new_tokens = []
            while i < len(tokens):
                if tokens[i] == val:
                    new_tokens.append(key[0])
                    new_tokens.append(key[1])
                    i += 1
                else:
                    new_tokens.append(tokens[i])
                    i += 1

            tokens = new_tokens
            
        byte_data = bytes(tokens)
        return byte_data.decode('utf-8', errors='replace')



    def save_merges_escaped(self, filepath):
        # 1. Reconstruct the full byte representation for every token ID.
        # Start with the base 256 bytes.
        vocab = {i: bytes([i]) for i in range(256)}
        
        # Reconstruct merged tokens in chronological order
        for (p0, p1), val in self.merges.items():
            vocab[val] = vocab[p0] + vocab[p1]
            
        # 2. Now write them using the reconstructed bytes
        with open(filepath, "w", encoding="utf-8") as f:
            for (p0, p1), val in self.merges.items():
                # Retrieve the underlying bytes for p0 and p1 from our vocab
                p0_bytes = vocab[p0]
                p1_bytes = vocab[p1]
                merged_bytes = vocab[val]
                
                # Safely decode them to strings and escape control characters
                p0_text = p0_bytes.decode("utf-8", errors="replace").encode("unicode_escape").decode("utf-8")
                p1_text = p1_bytes.decode("utf-8", errors="replace").encode("unicode_escape").decode("utf-8")
                merged_text = merged_bytes.decode("utf-8", errors="replace").encode("unicode_escape").decode("utf-8")
                
                f.write(f"[{p0_text}] + [{p1_text}] -> [{merged_text}] {val}\n")

In [8]:
regEncoder = RegexTokenizer()
with open("taylorswift.txt", "r", encoding="utf-8") as f:
    text = f.read()
regEncoder.train(text, 1000)

In [9]:
enc = tiktoken.get_encoding("cl100k_base") # this is the GPT-4 tokenizer
ids = enc.encode("hello world!!!? (안녕하세요!) lol123 😉")
print(ids)
text = enc.decode(ids) # get the same text back

print(text)

[15339, 1917, 12340, 30, 320, 31495, 230, 75265, 243, 92245, 16715, 28509, 4513, 57037]
hello world!!!? (안녕하세요!) lol123 😉


In [10]:
ids2 = regEncoder.encode("hello world!!!? (안녕하세요!) lol123 😉")
print(ids2)
text2 = regEncoder.decode(ids2)

print(text2)

[263, 300, 111, 337, 600, 33, 33, 33, 63, 293, 236, 149, 136, 235, 133, 149, 237, 149, 152, 236, 132, 184, 236, 154, 148, 33, 41, 442, 412, 49, 481, 32, 240, 159, 152, 137]
hello world!!!? (안녕하세요!) lol123 😉


In [ ]:
%pip install sentencepiece

In [12]:
import sentencepiece as spm

In [14]:
with open("toy.txt", "w", encoding="utf-8") as f:
    f.write("SentencePiece is a fast, lightweight, and unsupervised text tokenizer and detokenizer designed for neural network-based text generation systems (such as Large Language Models) where the vocabulary size is fixed prior to training.")

In [17]:
# train a sentencepiece model on it
# the settings here are (best effort) those used for training Llama 2
import os

options = dict(
  # input spec
  input="toy.txt",
  input_format="text",
  # output spec
  model_prefix="tok400", # output filename prefix
  # algorithm spec
  # BPE alg
  model_type="bpe",
  vocab_size=400,
  # normalization
  normalization_rule_name="identity", # ew, turn off normalization
  remove_extra_whitespaces=False,
  input_sentence_size=200000000, # max number of training sentences
  max_sentence_length=4192, # max number of bytes per sentence
  seed_sentencepiece_size=1000000,
  shuffle_input_sentence=True,
  # rare word treatment
  character_coverage=0.99995,
  byte_fallback=True,
  # merge rules
  split_digits=True,
  split_by_unicode_script=True,
  split_by_whitespace=True,
  split_by_number=True,
  max_sentencepiece_length=16,
  add_dummy_prefix=True,
  allow_whitespace_only_pieces=True,
  # special tokens
  unk_id=0, # the UNK token MUST exist
  bos_id=1, # the others are optional, set to -1 to turn off
  eos_id=2,
  pad_id=-1,
  # systems
  num_threads=os.cpu_count(), # use ~all system resources
)

spm.SentencePieceTrainer.train(**options)


I0000 00:00:1784190253.475354 2597420 sentencepiece_trainer.cc:105] Starts training with : 
trainer_spec {
  input: toy.txt
  input_format: text
  model_prefix: tok400
  model_type: BPE
  vocab_size: 400
  self_test_sample_size: 0
  character_coverage: 0.99995
  input_sentence_size: 200000000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 10
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 1
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_d

True

In [18]:
sp = spm.SentencePieceProcessor()
sp.load('tok400.model')
vocab = [[sp.id_to_piece(idx), idx] for idx in range(sp.get_piece_size())]
vocab

[['<unk>', 0],
 ['<s>', 1],
 ['</s>', 2],
 ['<0x00>', 3],
 ['<0x01>', 4],
 ['<0x02>', 5],
 ['<0x03>', 6],
 ['<0x04>', 7],
 ['<0x05>', 8],
 ['<0x06>', 9],
 ['<0x07>', 10],
 ['<0x08>', 11],
 ['<0x09>', 12],
 ['<0x0A>', 13],
 ['<0x0B>', 14],
 ['<0x0C>', 15],
 ['<0x0D>', 16],
 ['<0x0E>', 17],
 ['<0x0F>', 18],
 ['<0x10>', 19],
 ['<0x11>', 20],
 ['<0x12>', 21],
 ['<0x13>', 22],
 ['<0x14>', 23],
 ['<0x15>', 24],
 ['<0x16>', 25],
 ['<0x17>', 26],
 ['<0x18>', 27],
 ['<0x19>', 28],
 ['<0x1A>', 29],
 ['<0x1B>', 30],
 ['<0x1C>', 31],
 ['<0x1D>', 32],
 ['<0x1E>', 33],
 ['<0x1F>', 34],
 ['<0x20>', 35],
 ['<0x21>', 36],
 ['<0x22>', 37],
 ['<0x23>', 38],
 ['<0x24>', 39],
 ['<0x25>', 40],
 ['<0x26>', 41],
 ['<0x27>', 42],
 ['<0x28>', 43],
 ['<0x29>', 44],
 ['<0x2A>', 45],
 ['<0x2B>', 46],
 ['<0x2C>', 47],
 ['<0x2D>', 48],
 ['<0x2E>', 49],
 ['<0x2F>', 50],
 ['<0x30>', 51],
 ['<0x31>', 52],
 ['<0x32>', 53],
 ['<0x33>', 54],
 ['<0x34>', 55],
 ['<0x35>', 56],
 ['<0x36>', 57],
 ['<0x37>', 58],
 ['<0x38>', 5

In [19]:
ids = sp.encode("hello world!!!? (안녕하세요!) lol123 😉")
print(ids)

[366, 302, 380, 380, 375, 366, 383, 268, 380, 374, 36, 36, 36, 66, 314, 239, 152, 139, 238, 136, 152, 240, 152, 155, 239, 135, 187, 239, 157, 151, 36, 393, 317, 375, 380, 52, 53, 54, 366, 243, 162, 155, 140]


In [20]:
print([sp.id_to_piece(idx) for idx in ids])

['▁', 'he', 'l', 'l', 'o', '▁', 'w', 'or', 'l', 'd', '<0x21>', '<0x21>', '<0x21>', '<0x3F>', '▁(', '<0xEC>', '<0x95>', '<0x88>', '<0xEB>', '<0x85>', '<0x95>', '<0xED>', '<0x95>', '<0x98>', '<0xEC>', '<0x84>', '<0xB8>', '<0xEC>', '<0x9A>', '<0x94>', '<0x21>', ')', '▁l', 'o', 'l', '<0x31>', '<0x32>', '<0x33>', '▁', '<0xF0>', '<0x9F>', '<0x98>', '<0x89>']
